# Match Music — Casos de prueba del cliente (Grupo B)

**Análisis de Datos II · UNICABA 2026.2** · Proveedor: Grupo A · Cliente: Grupo B

Este notebook ejecuta sobre el motor de inferencia la batería oficial de casos de prueba que entregó el Grupo B en la reunión 2 (22/10/2026), y muestra para cada caso:

1. la **entrada** tal como la define el cliente,
2. el **resultado esperado** según su ficha,
3. lo que **devuelve el sistema** (recomendaciones, artistas, mensaje de cierre),
4. la **traza**: reglas disparadas y cadena de cada recomendación,
5. la **verificación automática** del criterio de aceptación del cliente (OK / NO OK).

Los casos están definidos en `casos_prueba.py`; el mismo archivo lo usa `demo.py` para ejecutarlos por consola.
El notebook no contiene reglas: llama a `src.engine.recomendar()`, igual que la interfaz.

**Cómo ejecutarlo:** desde la raíz del repositorio, con el entorno instalado según el README,
`python -m pip install notebook` y luego `python -m jupyter notebook casos_cliente.ipynb`
(o abrirlo en VS Code). *Kernel → Restart & Run All*.

In [1]:
import platform

from IPython.display import Markdown, display

from casos_prueba import CASOS, ejecutar
from src.engine import REGLAS, cargar_catalogo, preparar_preferencias
from src.tracer import NIVELES, agrupar_por_nivel, presentar

CATALOGO = cargar_catalogo()
print(f"Python {platform.python_version()} · {len(CATALOGO)} canciones en el catálogo · "
      f"{len(REGLAS)} reglas en la base de conocimiento")

Python 3.13.18 · 34 canciones en el catálogo · 41 reglas en la base de conocimiento


In [2]:
def tabla(filas, columnas):
    """Tabla Markdown a partir de una lista de tuplas."""
    lineas = ["| " + " | ".join(columnas) + " |", "|" + "---|" * len(columnas)]
    lineas += ["| " + " | ".join(str(c).replace("|", "/") for c in f) + " |" for f in filas]
    return "\n".join(lineas)


def mostrar_caso(caso, traza_completa=False):
    """Ejecuta un caso y muestra entrada, esperado, resultado, traza y verificación."""
    resultado, checks, aprobado = ejecutar(caso)
    datos = presentar(resultado, CATALOGO)

    partes = [
        f"**Requisitos:** {caso['requisitos']} · **Prioridad:** {caso['prioridad']}",
        f"**Entrada:** `{caso['entrada']}` → hechos de entrada para el motor: "
        f"`UserPreferences({preparar_preferencias(caso['entrada'])})`",
        f"**Resultado esperado (ficha del cliente):** {caso['esperado']}",
    ]
    if datos["sin_conclusion"]:
        partes.append(f"**Resultado del sistema:** ⚠ {datos['sin_conclusion']}")
    elif datos["sin_coincidencia"]:
        partes.append(f"**Resultado del sistema:** ⚠ {datos['sin_coincidencia']}")
    else:
        partes.append(f"**Resultado del sistema:** {len(datos['recomendaciones'])} canciones · "
                      f"artistas recomendados: {', '.join(datos['artistas'])}")
        partes.append(tabla(
            [(i, f"{r['titulo']} — {r['artista']}", r["anio"], r["pais"], r["puntaje"],
              "; ".join(r["motivos"]), " → ".join(c["id"] for c in r["cadena"]))
             for i, r in enumerate(datos["recomendaciones"], start=1)],
            ["#", "Canción", "Año", "País", "Puntaje", "Motivos", "Cadena de reglas"]))

    grupos = agrupar_por_nivel(datos["traza"])
    partes.append(f"**Traza:** {len(datos['traza'])} disparos de reglas, por nivel:")
    if traza_completa:
        for nivel, pasos in grupos.items():
            partes.append(f"*{NIVELES[nivel]}*")
            partes.append(tabla([(p["paso"], p["regla"], p["nombre"], p["descripcion"]) for p in pasos],
                                ["Paso", "Regla", "Nombre", "Descripción"]))
    else:
        partes.append(tabla(
            [(NIVELES[n], len(p), ", ".join(sorted({x["regla"] + " " + x["nombre"] for x in p})))
             for n, p in grupos.items()],
            ["Nivel", "Disparos", "Reglas"]))

    partes.append("**Verificación del criterio de aceptación:**")
    partes.append("\n".join(f"- {'✔' if ok else '✘'} {d}" for d, ok in checks))
    partes.append(f"### Resultado: {'✅ OK' if aprobado else '❌ NO OK'}")
    display(Markdown("\n\n".join(partes)))
    return resultado, aprobado


resultados = {}

## CA-01 — Recomendación por estilo y país (rock nacional)

In [3]:
resultados["CA-01"] = mostrar_caso(CASOS[0])

**Requisitos:** RF-01 / RF-03 · **Prioridad:** MUST

**Entrada:** `{'generos': ['Rock nacional'], 'pais': 'Argentina'}` → hechos de entrada para el motor: `UserPreferences({'generos': ('rock nacional',), 'pais': 'argentina'})`

**Resultado esperado (ficha del cliente):** Al menos una canción o artista referente (ej. Charly García, Soda Stereo, Spinetta). Coincidencia directa por género y origen.

**Resultado del sistema:** 8 canciones · artistas recomendados: Babasónicos, Charly García, Gustavo Cerati, Los Abuelos de la Nada, Luis Alberto Spinetta, Soda Stereo, Virus

| # | Canción | Año | País | Puntaje | Motivos | Cadena de reglas |
|---|---|---|---|---|---|---|
| 1 | Demoliendo hoteles — Charly García | 1984 | Argentina | 6 | pertenencia al contexto «Música latina»; subgénero «Rock nacional»; país «Argentina» | R01 → R04 → R09 → R12 → R15 → R17 → R22 → R23 |
| 2 | Mil horas — Los Abuelos de la Nada | 1983 | Argentina | 6 | pertenencia al contexto «Música latina»; subgénero «Rock nacional»; país «Argentina» | R01 → R04 → R09 → R12 → R15 → R17 → R22 → R23 |
| 3 | Seguir viviendo sin tu amor — Luis Alberto Spinetta | 1986 | Argentina | 6 | pertenencia al contexto «Música latina»; subgénero «Rock nacional»; país «Argentina» | R01 → R04 → R09 → R12 → R15 → R17 → R22 → R23 |
| 4 | Crimen — Gustavo Cerati | 2006 | Argentina | 4 | pertenencia al contexto «Música latina»; país «Argentina»; subgénero «Pop rock», vecino de «Rock nacional» | R01 → R04 → R09 → R15 → R17 → R31 → R35 → R22 → R23 |
| 5 | De música ligera — Soda Stereo | 1990 | Argentina | 4 | pertenencia al contexto «Música latina»; país «Argentina»; subgénero «Rock en español», vecino de «Rock nacional» | R01 → R04 → R09 → R15 → R17 → R31 → R35 → R22 → R23 |
| 6 | Irresponsables — Babasónicos | 2003 | Argentina | 4 | pertenencia al contexto «Música latina»; país «Argentina»; género «Rock», que incluye al estilo pedido «Rock nacional» | R01 → R04 → R09 → R13 → R15 → R17 → R30 → R22 → R23 |
| 7 | Persiana americana — Soda Stereo | 1986 | Argentina | 4 | pertenencia al contexto «Música latina»; país «Argentina»; subgénero «New wave», vecino de «Rock nacional» | R01 → R04 → R09 → R15 → R17 → R31 → R35 → R22 → R23 |
| 8 | Una luna de miel en la mano — Virus | 1985 | Argentina | 4 | pertenencia al contexto «Música latina»; país «Argentina»; subgénero «New wave», vecino de «Rock nacional» | R01 → R04 → R09 → R15 → R17 → R31 → R35 → R22 → R23 |

**Traza:** 110 disparos de reglas, por nivel:

| Nivel | Disparos | Reglas |
|---|---|---|
| Nivel 1A: lectura de tus preferencias | 2 | R01 Estilos pedidos, R04 País pedido |
| Nivel 1B: contexto musical deducido | 3 | R09 Contexto latino por país, R30 Género que contiene al subgénero pedido, R31 Subgéneros vecinos del estilo pedido |
| Nivel 2: comparación con el catálogo | 49 | R12 Coincide el subgénero, R13 Coincide un género relacionado, R15 Coincide el país, R17 Encaja en música latina, R35 Coincide un subgénero vecino |
| Nivel 2B: filtro por estilo | 8 | R21 Descarte por estilo |
| Nivel 3: puntaje y decisión | 36 | R22 Puntaje consolidado, R23 Recomendación emitida, R24 Puntaje insuficiente |
| Nivel 4: artistas y estilos sugeridos | 12 | R25 Artista recomendado, R26 Estilo sugerido |

**Verificación del criterio de aceptación:**

- ✔ Devuelve al menos una recomendación
- ✔ Incluye referentes del rock nacional (Charly García o Soda Stereo)
- ✔ Todas las recomendaciones son de Argentina
- ✔ La traza muestra coincidencia directa por estilo (R12) y por país (R15)

### Resultado: ✅ OK

## CA-02 — Artistas similares a una banda base

In [4]:
resultados["CA-02"] = mostrar_caso(CASOS[1])

**Requisitos:** RF-02 / RF-08 · **Prioridad:** MUST

**Entrada:** `{'artista': 'Soda Stereo'}` → hechos de entrada para el motor: `UserPreferences({'artista': 'soda stereo'})`

**Resultado esperado (ficha del cliente):** Artistas compatibles (ej. Virus, Babasónicos, Zoé), vinculados por subgénero pop rock / alternativo.

**Resultado del sistema:** 10 canciones · artistas recomendados: Babasónicos, Charly García, Gustavo Cerati, Jarabe de Palo, Los Abuelos de la Nada, Los Prisioneros, Luis Alberto Spinetta, Nena, Virus, Zoé

| # | Canción | Año | País | Puntaje | Motivos | Cadena de reglas |
|---|---|---|---|---|---|---|
| 1 | Tren al sur — Los Prisioneros | 1990 | Chile | 4 | mismo idioma que la referencia («Español»); mismo subgénero que «Soda Stereo» («Rock en español») | R28 → R32 → R36 → R38 → R22 → R23 |
| 2 | Una luna de miel en la mano — Virus | 1985 | Argentina | 4 | mismo idioma que la referencia («Español»); mismo subgénero que «Soda Stereo» («New wave») | R28 → R32 → R36 → R38 → R22 → R23 |
| 3 | 99 Luftballons — Nena | 1983 | Alemania | 3 | mismo subgénero que «Soda Stereo» («New wave») | R28 → R32 → R36 → R22 → R23 |
| 4 | Crimen — Gustavo Cerati | 2006 | Argentina | 3 | mismo idioma que la referencia («Español»); subgénero «Pop rock», vecino de «Rock en español» de la referencia | R28 → R32 → R34 → R37 → R38 → R22 → R23 |
| 5 | Demoliendo hoteles — Charly García | 1984 | Argentina | 3 | subgénero «Rock nacional», vecino de «New wave» de la referencia; mismo idioma que la referencia («Español») | R28 → R32 → R34 → R37 → R38 → R22 → R23 |
| 6 | Irresponsables — Babasónicos | 2003 | Argentina | 3 | mismo idioma que la referencia («Español»); subgénero «Rock alternativo», vecino de «Rock en español» de la referencia | R28 → R32 → R34 → R37 → R38 → R22 → R23 |
| 7 | La flaca — Jarabe de Palo | 1996 | España | 3 | mismo idioma que la referencia («Español»); subgénero «Pop rock», vecino de «Rock en español» de la referencia | R28 → R32 → R34 → R37 → R38 → R22 → R23 |
| 8 | Mil horas — Los Abuelos de la Nada | 1983 | Argentina | 3 | subgénero «Rock nacional», vecino de «New wave» de la referencia; mismo idioma que la referencia («Español») | R28 → R32 → R34 → R37 → R38 → R22 → R23 |
| 9 | Seguir viviendo sin tu amor — Luis Alberto Spinetta | 1986 | Argentina | 3 | subgénero «Rock nacional», vecino de «New wave» de la referencia; mismo idioma que la referencia («Español») | R28 → R32 → R34 → R37 → R38 → R22 → R23 |
| 10 | Soñé — Zoé | 2003 | México | 3 | mismo idioma que la referencia («Español»); subgénero «Rock alternativo», vecino de «Rock en español» de la referencia | R28 → R32 → R34 → R37 → R38 → R22 → R23 |

**Traza:** 96 disparos de reglas, por nivel:

| Nivel | Disparos | Reglas |
|---|---|---|
| Nivel 1A: lectura de tus preferencias | 1 | R28 Artista de referencia |
| Nivel 1B: contexto musical deducido | 4 | R32 Rasgos del artista de referencia, R34 Estilos vecinos de la referencia |
| Nivel 2: comparación con el catálogo | 35 | R36 Mismo estilo que la referencia, R37 Estilo vecino de la referencia, R38 Mismo idioma que la referencia, R39 Descarte: es el artista de referencia |
| Nivel 3: puntaje y decisión | 40 | R22 Puntaje consolidado, R23 Recomendación emitida, R24 Puntaje insuficiente |
| Nivel 4: artistas y estilos sugeridos | 16 | R25 Artista recomendado, R26 Estilo sugerido |

**Verificación del criterio de aceptación:**

- ✔ Devuelve Virus, Babasónicos y Zoé
- ✔ No recomienda al propio artista de referencia
- ✔ La traza muestra la relación por subgénero (R36 / R37)

### Resultado: ✅ OK

## CA-03 — Filtrado por época (años 80)

In [5]:
resultados["CA-03"] = mostrar_caso(CASOS[2])

**Requisitos:** RF-04 · **Prioridad:** SHOULD

**Entrada:** `{'generos': ['Pop', 'Synth-pop'], 'epoca': '80s', 'idioma': 'Inglés'}` → hechos de entrada para el motor: `UserPreferences({'generos': ('pop', 'synth-pop'), 'epoca': '80s', 'idioma': 'ingles'})`

**Resultado esperado (ficha del cliente):** Artistas o temas de 1980 a 1989 (ej. Michael Jackson, Madonna, Depeche Mode). Excluye lanzamientos posteriores a 1989.

**Resultado del sistema:** 6 canciones · artistas recomendados: Bon Jovi, Depeche Mode, Guns N' Roses, Madonna, Michael Jackson, a-ha

| # | Canción | Año | País | Puntaje | Motivos | Cadena de reglas |
|---|---|---|---|---|---|---|
| 1 | Just Can't Get Enough — Depeche Mode | 1981 | Reino Unido | 10 | subgénero «Synth-pop»; género «Pop»; año 1981 dentro de 1980–1989; idioma «Inglés» | R01 → R02 → R06 → R11 → R12 → R14 → R16 → R22 → R23 |
| 2 | Take On Me — a-ha | 1985 | Noruega | 10 | subgénero «Synth-pop»; género «Pop»; año 1985 dentro de 1980–1989; idioma «Inglés» | R01 → R02 → R06 → R11 → R12 → R14 → R16 → R22 → R23 |
| 3 | Like a Virgin — Madonna | 1984 | Estados Unidos | 8 | género «Pop»; año 1984 dentro de 1980–1989; idioma «Inglés»; subgénero «Dance-pop», vecino de «Synth-pop» | R01 → R02 → R06 → R11 → R14 → R16 → R31 → R35 → R22 → R23 |
| 4 | Billie Jean — Michael Jackson | 1982 | Estados Unidos | 7 | género «Pop»; año 1982 dentro de 1980–1989; idioma «Inglés» | R01 → R02 → R06 → R11 → R14 → R16 → R22 → R23 |
| 5 | Livin' on a Prayer — Bon Jovi | 1986 | Estados Unidos | 5 | año 1986 dentro de 1980–1989; idioma «Inglés»; género «Rock», afín a «Pop» | R01 → R02 → R06 → R07 → R13 → R14 → R16 → R22 → R23 |
| 6 | Sweet Child O' Mine — Guns N' Roses | 1987 | Estados Unidos | 5 | año 1987 dentro de 1980–1989; idioma «Inglés»; género «Rock», afín a «Pop» | R01 → R02 → R06 → R07 → R13 → R14 → R16 → R22 → R23 |

**Traza:** 115 disparos de reglas, por nivel:

| Nivel | Disparos | Reglas |
|---|---|---|
| Nivel 1A: lectura de tus preferencias | 4 | R01 Estilos pedidos, R02 Era convertida en período, R06 Idioma pedido |
| Nivel 1B: contexto musical deducido | 2 | R07 Géneros afines, R31 Subgéneros vecinos del estilo pedido |
| Nivel 2: comparación con el catálogo | 86 | R11 Coincide el género, R12 Coincide el subgénero, R13 Coincide un género relacionado, R14 Coincide la época, R16 Coincide el idioma, R19 Descarte por idioma, R20 Descarte por época, R35 Coincide un subgénero vecino |
| Nivel 2B: filtro por estilo | 1 | R21 Descarte por estilo |
| Nivel 3: puntaje y decisión | 12 | R22 Puntaje consolidado, R23 Recomendación emitida |
| Nivel 4: artistas y estilos sugeridos | 10 | R25 Artista recomendado, R26 Estilo sugerido |

**Verificación del criterio de aceptación:**

- ✔ Devuelve recomendaciones
- ✔ Todas son de 1980–1989
- ✔ Incluye a Depeche Mode y Michael Jackson
- ✔ La traza muestra la restricción temporal (R02 → R14 / R20)

### Resultado: ✅ OK

## CA-04 — Música local e idioma

In [6]:
resultados["CA-04"] = mostrar_caso(CASOS[3])

**Requisitos:** RF-06 / RF-07 · **Prioridad:** SHOULD

**Entrada:** `{'generos': ['Indie', 'Folk'], 'idioma': 'Español', 'local': True}` → hechos de entrada para el motor: `UserPreferences({'generos': ('indie', 'folk'), 'local': True, 'idioma': 'espanol'})`

**Resultado esperado (ficha del cliente):** Bandas locales en español (ej. Él Mató a un Policía Motorizado, Bandalos Chinos), con incremento de puntaje por idioma y música local.

**Resultado del sistema:** 16 canciones · artistas recomendados: Babasónicos, Bandalos Chinos, Carlos Vives y Shakira, Charly García, Gustavo Cerati, Jarabe de Palo, Juana Molina, Los Abuelos de la Nada, Los Prisioneros, Luis Alberto Spinetta, Mercedes Sosa, Soda Stereo, Virus, Zoé, Él Mató a un Policía Motorizado

| # | Canción | Año | País | Puntaje | Motivos | Cadena de reglas |
|---|---|---|---|---|---|---|
| 1 | Mujeres bellas y fuertes — Él Mató a un Policía Motorizado | 2012 | Argentina | 8 | pertenencia al contexto «Música latina»; país «Argentina»; género «Indie»; idioma «Español» | R01 → R05 → R06 → R09 → R11 → R15 → R16 → R17 → R22 → R23 |
| 2 | Un día — Juana Molina | 2008 | Argentina | 8 | pertenencia al contexto «Música latina»; país «Argentina»; género «Folk»; idioma «Español» | R01 → R05 → R06 → R09 → R11 → R15 → R16 → R17 → R22 → R23 |
| 3 | Vámonos de viaje — Bandalos Chinos | 2018 | Argentina | 8 | pertenencia al contexto «Música latina»; país «Argentina»; género «Indie»; idioma «Español» | R01 → R05 → R06 → R09 → R11 → R15 → R16 → R17 → R22 → R23 |
| 4 | Crimen — Gustavo Cerati | 2006 | Argentina | 6 | pertenencia al contexto «Música latina»; país «Argentina»; idioma «Español»; género «Rock», afín a «Indie» | R01 → R05 → R06 → R07 → R09 → R13 → R15 → R16 → R17 → R22 → R23 |
| 5 | De música ligera — Soda Stereo | 1990 | Argentina | 6 | pertenencia al contexto «Música latina»; país «Argentina»; idioma «Español»; género «Rock», afín a «Indie» | R01 → R05 → R06 → R07 → R09 → R13 → R15 → R16 → R17 → R22 → R23 |
| 6 | Demoliendo hoteles — Charly García | 1984 | Argentina | 6 | pertenencia al contexto «Música latina»; país «Argentina»; idioma «Español»; género «Rock», afín a «Indie» | R01 → R05 → R06 → R07 → R09 → R13 → R15 → R16 → R17 → R22 → R23 |
| 7 | Gracias a la vida — Mercedes Sosa | 1971 | Argentina | 6 | pertenencia al contexto «Música latina»; país «Argentina»; idioma «Español»; género «Folklore», afín a «Folk» | R01 → R05 → R06 → R07 → R09 → R13 → R15 → R16 → R17 → R22 → R23 |
| 8 | Irresponsables — Babasónicos | 2003 | Argentina | 6 | pertenencia al contexto «Música latina»; país «Argentina»; idioma «Español»; género «Rock», afín a «Indie» | R01 → R05 → R06 → R07 → R09 → R13 → R15 → R16 → R17 → R22 → R23 |
| 9 | Mil horas — Los Abuelos de la Nada | 1983 | Argentina | 6 | pertenencia al contexto «Música latina»; país «Argentina»; idioma «Español»; género «Rock», afín a «Indie» | R01 → R05 → R06 → R07 → R09 → R13 → R15 → R16 → R17 → R22 → R23 |
| 10 | Persiana americana — Soda Stereo | 1986 | Argentina | 6 | pertenencia al contexto «Música latina»; país «Argentina»; idioma «Español»; género «Rock», afín a «Indie» | R01 → R05 → R06 → R07 → R09 → R13 → R15 → R16 → R17 → R22 → R23 |
| 11 | Seguir viviendo sin tu amor — Luis Alberto Spinetta | 1986 | Argentina | 6 | pertenencia al contexto «Música latina»; país «Argentina»; idioma «Español»; género «Rock», afín a «Indie» | R01 → R05 → R06 → R07 → R09 → R13 → R15 → R16 → R17 → R22 → R23 |
| 12 | Una luna de miel en la mano — Virus | 1985 | Argentina | 6 | pertenencia al contexto «Música latina»; país «Argentina»; idioma «Español»; género «Rock», afín a «Indie» | R01 → R05 → R06 → R07 → R09 → R13 → R15 → R16 → R17 → R22 → R23 |
| 13 | La Bicicleta — Carlos Vives y Shakira | 2016 | Colombia | 4 | pertenencia al contexto «Música latina»; idioma «Español»; género «Pop», afín a «Indie» | R01 → R05 → R06 → R07 → R09 → R13 → R16 → R17 → R22 → R23 |
| 14 | Soñé — Zoé | 2003 | México | 4 | pertenencia al contexto «Música latina»; idioma «Español»; género «Rock», afín a «Indie» | R01 → R05 → R06 → R07 → R09 → R13 → R16 → R17 → R22 → R23 |
| 15 | Tren al sur — Los Prisioneros | 1990 | Chile | 4 | pertenencia al contexto «Música latina»; idioma «Español»; género «Rock», afín a «Indie» | R01 → R05 → R06 → R07 → R09 → R13 → R16 → R17 → R22 → R23 |
| 16 | La flaca — Jarabe de Palo | 1996 | España | 3 | idioma «Español»; género «Rock», afín a «Indie» | R01 → R06 → R07 → R13 → R16 → R22 → R23 |

**Traza:** 160 disparos de reglas, por nivel:

| Nivel | Disparos | Reglas |
|---|---|---|
| Nivel 1A: lectura de tus preferencias | 4 | R01 Estilos pedidos, R05 Música local como país local, R06 Idioma pedido |
| Nivel 1B: contexto musical deducido | 3 | R07 Géneros afines, R09 Contexto latino por país |
| Nivel 2: comparación con el catálogo | 94 | R11 Coincide el género, R13 Coincide un género relacionado, R15 Coincide el país, R16 Coincide el idioma, R17 Encaja en música latina, R19 Descarte por idioma |
| Nivel 2B: filtro por estilo | 2 | R21 Descarte por estilo |
| Nivel 3: puntaje y decisión | 32 | R22 Puntaje consolidado, R23 Recomendación emitida |
| Nivel 4: artistas y estilos sugeridos | 25 | R25 Artista recomendado, R26 Estilo sugerido |

**Verificación del criterio de aceptación:**

- ✔ Devuelve recomendaciones
- ✔ Las primeras posiciones son indie o folk local
- ✔ Todas las recomendaciones son en español
- ✔ La traza muestra la ponderación por música local e idioma (R05, R15, R16)

### Resultado: ✅ OK

## CA-05 — Consulta multiatributo

In [7]:
resultados["CA-05"] = mostrar_caso(CASOS[4])

**Requisitos:** RF-01 / RF-03 / RF-05 · **Prioridad:** MUST

**Entrada:** `{'generos': ['Hard rock'], 'pais': 'Reino Unido', 'epoca': '70s', 'idioma': 'Inglés'}` → hechos de entrada para el motor: `UserPreferences({'generos': ('hard rock',), 'epoca': '70s', 'pais': 'reino unido', 'idioma': 'ingles'})`

**Resultado esperado (ficha del cliente):** Led Zeppelin, Black Sabbath, Deep Purple. Encadenamiento hacia adelante de al menos 3 niveles.

**Resultado del sistema:** 4 canciones · artistas recomendados: Black Sabbath, Deep Purple, Led Zeppelin, Queen

| # | Canción | Año | País | Puntaje | Motivos | Cadena de reglas |
|---|---|---|---|---|---|---|
| 1 | Black Dog — Led Zeppelin | 1971 | Reino Unido | 9 | subgénero «Hard rock»; año 1971 dentro de 1970–1979; país «Reino Unido»; idioma «Inglés» | R01 → R02 → R04 → R06 → R12 → R14 → R15 → R16 → R22 → R23 |
| 2 | Smoke on the Water — Deep Purple | 1972 | Reino Unido | 9 | subgénero «Hard rock»; año 1972 dentro de 1970–1979; país «Reino Unido»; idioma «Inglés» | R01 → R02 → R04 → R06 → R12 → R14 → R15 → R16 → R22 → R23 |
| 3 | Bohemian Rhapsody — Queen | 1975 | Reino Unido | 7 | año 1975 dentro de 1970–1979; país «Reino Unido»; idioma «Inglés»; género «Rock», que incluye al estilo pedido «Hard rock» | R01 → R02 → R04 → R06 → R13 → R14 → R15 → R16 → R30 → R22 → R23 |
| 4 | Paranoid — Black Sabbath | 1970 | Reino Unido | 7 | año 1970 dentro de 1970–1979; país «Reino Unido»; idioma «Inglés»; subgénero «Heavy metal», vecino de «Hard rock» | R01 → R02 → R04 → R06 → R14 → R15 → R16 → R31 → R35 → R22 → R23 |

**Traza:** 97 disparos de reglas, por nivel:

| Nivel | Disparos | Reglas |
|---|---|---|
| Nivel 1A: lectura de tus preferencias | 4 | R01 Estilos pedidos, R02 Era convertida en período, R04 País pedido, R06 Idioma pedido |
| Nivel 1B: contexto musical deducido | 2 | R30 Género que contiene al subgénero pedido, R31 Subgéneros vecinos del estilo pedido |
| Nivel 2: comparación con el catálogo | 76 | R12 Coincide el subgénero, R13 Coincide un género relacionado, R14 Coincide la época, R15 Coincide el país, R16 Coincide el idioma, R19 Descarte por idioma, R20 Descarte por época, R35 Coincide un subgénero vecino |
| Nivel 3: puntaje y decisión | 8 | R22 Puntaje consolidado, R23 Recomendación emitida |
| Nivel 4: artistas y estilos sugeridos | 7 | R25 Artista recomendado, R26 Estilo sugerido |

**Verificación del criterio de aceptación:**

- ✔ Devuelve Led Zeppelin, Black Sabbath y Deep Purple
- ✔ Todas son de Reino Unido, años 70 y en inglés
- ✔ La cadena de «Black Dog» encadena al menos 3 niveles (1A → 2 → 3)

### Resultado: ✅ OK

## CA-06 — Búsqueda sin coincidencias (caso límite)

In [8]:
resultados["CA-06"] = mostrar_caso(CASOS[5])

**Requisitos:** RF-01 / RF-07 · **Prioridad:** Caso límite

**Entrada:** `{'generos': ['Heavy metal'], 'pais': 'Japón', 'epoca': '20s'}` → hechos de entrada para el motor: `UserPreferences({'generos': ('heavy metal',), 'epoca': '20s', 'pais': 'japon'})`

**Resultado esperado (ficha del cliente):** No falla. Emite: «No se encontraron recomendaciones con coincidencia suficiente para los criterios seleccionados. Modifique o amplíe los criterios de búsqueda.»

**Resultado del sistema:** ⚠ No se encontraron recomendaciones con coincidencia suficiente para los criterios seleccionados. Modifique o amplíe los criterios de búsqueda.

**Traza:** 46 disparos de reglas, por nivel:

| Nivel | Disparos | Reglas |
|---|---|---|
| Nivel 1A: lectura de tus preferencias | 3 | R01 Estilos pedidos, R02 Era convertida en período, R04 País pedido |
| Nivel 1B: contexto musical deducido | 2 | R30 Género que contiene al subgénero pedido, R31 Subgéneros vecinos del estilo pedido |
| Nivel 2: comparación con el catálogo | 40 | R12 Coincide el subgénero, R20 Descarte por época, R35 Coincide un subgénero vecino |
| Cierre: sin coincidencia suficiente | 1 | R27 Sin coincidencia suficiente |

**Verificación del criterio de aceptación:**

- ✔ No devuelve recomendaciones
- ✔ Emite el mensaje literal del Brief §7
- ✔ El catálogo se evaluó completo y cerró con R27

### Resultado: ✅ OK

## CA-07 — Entrada vacía (caso límite)

In [9]:
resultados["CA-07"] = mostrar_caso(CASOS[6])

**Requisitos:** RF-01 · **Prioridad:** Caso límite

**Entrada:** `{}` → hechos de entrada para el motor: `UserPreferences({})`

**Resultado esperado (ficha del cliente):** Advertencia: «Sin conclusión posible: Ingrese al menos un criterio (estilo, artista, década o país) para ejecutar la recomendación.»

**Resultado del sistema:** ⚠ Sin conclusión posible: Ingrese al menos un criterio (estilo, artista, década o país) para ejecutar la recomendación.

**Traza:** 1 disparos de reglas, por nivel:

| Nivel | Disparos | Reglas |
|---|---|---|
| Nivel 0: validación de la consulta | 1 | R00 Validación de datos mínimos |

**Verificación del criterio de aceptación:**

- ✔ Emite el mensaje literal del caso
- ✔ El motor se detiene antes de evaluar el catálogo (sólo dispara R00)
- ✔ No devuelve recomendaciones

### Resultado: ✅ OK

## CA-08 — Explicabilidad y exposición de la traza

Se ejecuta sobre la consulta de CA-05 y se muestra la traza **completa**: cada disparo de regla, en orden, agrupado por nivel de encadenamiento.

In [10]:
resultados["CA-08"] = mostrar_caso(CASOS[7], traza_completa=True)

**Requisitos:** RF-08 · **Prioridad:** MUST

**Entrada:** `{'generos': ['Hard rock'], 'pais': 'Reino Unido', 'epoca': '70s', 'idioma': 'Inglés'}` → hechos de entrada para el motor: `UserPreferences({'generos': ('hard rock',), 'epoca': '70s', 'pais': 'reino unido', 'idioma': 'ingles'})`

**Resultado esperado (ficha del cliente):** Bloque de inferencia con hechos de entrada, reglas disparadas con nombre y conclusión final.

**Resultado del sistema:** 4 canciones · artistas recomendados: Black Sabbath, Deep Purple, Led Zeppelin, Queen

| # | Canción | Año | País | Puntaje | Motivos | Cadena de reglas |
|---|---|---|---|---|---|---|
| 1 | Black Dog — Led Zeppelin | 1971 | Reino Unido | 9 | subgénero «Hard rock»; año 1971 dentro de 1970–1979; país «Reino Unido»; idioma «Inglés» | R01 → R02 → R04 → R06 → R12 → R14 → R15 → R16 → R22 → R23 |
| 2 | Smoke on the Water — Deep Purple | 1972 | Reino Unido | 9 | subgénero «Hard rock»; año 1972 dentro de 1970–1979; país «Reino Unido»; idioma «Inglés» | R01 → R02 → R04 → R06 → R12 → R14 → R15 → R16 → R22 → R23 |
| 3 | Bohemian Rhapsody — Queen | 1975 | Reino Unido | 7 | año 1975 dentro de 1970–1979; país «Reino Unido»; idioma «Inglés»; género «Rock», que incluye al estilo pedido «Hard rock» | R01 → R02 → R04 → R06 → R13 → R14 → R15 → R16 → R30 → R22 → R23 |
| 4 | Paranoid — Black Sabbath | 1970 | Reino Unido | 7 | año 1970 dentro de 1970–1979; país «Reino Unido»; idioma «Inglés»; subgénero «Heavy metal», vecino de «Hard rock» | R01 → R02 → R04 → R06 → R14 → R15 → R16 → R31 → R35 → R22 → R23 |

**Traza:** 97 disparos de reglas, por nivel:

*Nivel 1A: lectura de tus preferencias*

| Paso | Regla | Nombre | Descripción |
|---|---|---|---|
| 1 | R06 | Idioma pedido | El usuario busca canciones cantadas en «Inglés». |
| 2 | R04 | País pedido | El usuario busca música de «Reino Unido». |
| 3 | R02 | Era convertida en período | La era «70s» equivale al período 1970–1979. |
| 4 | R01 | Estilos pedidos | El usuario pide el estilo «Hard rock». |

*Nivel 1B: contexto musical deducido*

| Paso | Regla | Nombre | Descripción |
|---|---|---|---|
| 5 | R30 | Género que contiene al subgénero pedido | «Hard rock» es un subgénero de «Rock» ⇒ «Rock» también es relevante. |
| 6 | R31 | Subgéneros vecinos del estilo pedido | «Hard rock» tiene subgéneros vecinos: grunge, heavy metal. |

*Nivel 2: comparación con el catálogo*

| Paso | Regla | Nombre | Descripción |
|---|---|---|---|
| 7 | R12 | Coincide el subgénero | [C32] coincide por subgénero «Hard rock» (+3). |
| 8 | R12 | Coincide el subgénero | [C31] coincide por subgénero «Hard rock» (+3). |
| 9 | R12 | Coincide el subgénero | [C07] coincide por subgénero «Hard rock» (+3). |
| 10 | R12 | Coincide el subgénero | [C06] coincide por subgénero «Hard rock» (+3). |
| 11 | R20 | Descarte por época | [C34] descartada: año 1982 fuera de 1970–1979. |
| 12 | R14 | Coincide la época | [C33] coincide por año 1970 dentro de 1970–1979 (+2). |
| 13 | R14 | Coincide la época | [C32] coincide por año 1972 dentro de 1970–1979 (+2). |
| 14 | R14 | Coincide la época | [C31] coincide por año 1971 dentro de 1970–1979 (+2). |
| 15 | R14 | Coincide la época | [C30] coincide por año 1971 dentro de 1970–1979 (+2). |
| 16 | R20 | Descarte por época | [C29] descartada: año 2008 fuera de 1970–1979. |
| 17 | R20 | Descarte por época | [C28] descartada: año 2018 fuera de 1970–1979. |
| 18 | R20 | Descarte por época | [C27] descartada: año 2012 fuera de 1970–1979. |
| 19 | R20 | Descarte por época | [C26] descartada: año 1984 fuera de 1970–1979. |
| 20 | R20 | Descarte por época | [C25] descartada: año 1981 fuera de 1970–1979. |
| 21 | R20 | Descarte por época | [C24] descartada: año 2006 fuera de 1970–1979. |
| 22 | R20 | Descarte por época | [C23] descartada: año 2003 fuera de 1970–1979. |
| 23 | R20 | Descarte por época | [C22] descartada: año 2003 fuera de 1970–1979. |
| 24 | R20 | Descarte por época | [C21] descartada: año 1985 fuera de 1970–1979. |
| 25 | R20 | Descarte por época | [C20] descartada: año 1986 fuera de 1970–1979. |
| 26 | R20 | Descarte por época | [C19] descartada: año 1984 fuera de 1970–1979. |
| 27 | R14 | Coincide la época | [C18] coincide por año 1974 dentro de 1970–1979 (+2). |
| 28 | R20 | Descarte por época | [C17] descartada: año 2019 fuera de 1970–1979. |
| 29 | R20 | Descarte por época | [C16] descartada: año 2017 fuera de 1970–1979. |
| 30 | R20 | Descarte por época | [C15] descartada: año 2016 fuera de 1970–1979. |
| 31 | R20 | Descarte por época | [C14] descartada: año 2019 fuera de 1970–1979. |
| 32 | R20 | Descarte por época | [C13] descartada: año 2010 fuera de 1970–1979. |
| 33 | R20 | Descarte por época | [C12] descartada: año 1983 fuera de 1970–1979. |
| 34 | R20 | Descarte por época | [C11] descartada: año 1985 fuera de 1970–1979. |
| 35 | R20 | Descarte por época | [C10] descartada: año 1982 fuera de 1970–1979. |
| 36 | R14 | Coincide la época | [C09] coincide por año 1975 dentro de 1970–1979 (+2). |
| 37 | R20 | Descarte por época | [C08] descartada: año 1991 fuera de 1970–1979. |
| 38 | R20 | Descarte por época | [C07] descartada: año 1986 fuera de 1970–1979. |
| 39 | R20 | Descarte por época | [C06] descartada: año 1987 fuera de 1970–1979. |
| 40 | R20 | Descarte por época | [C05] descartada: año 1996 fuera de 1970–1979. |
| 41 | R20 | Descarte por época | [C04] descartada: año 1990 fuera de 1970–1979. |
| 42 | R20 | Descarte por época | [C03] descartada: año 1983 fuera de 1970–1979. |
| 43 | R20 | Descarte por época | [C02] descartada: año 1990 fuera de 1970–1979. |
| 44 | R20 | Descarte por época | [C01] descartada: año 1986 fuera de 1970–1979. |
| 45 | R15 | Coincide el país | [C34] coincide por país «Reino Unido» (+2). |
| 46 | R15 | Coincide el país | [C33] coincide por país «Reino Unido» (+2). |
| 47 | R15 | Coincide el país | [C32] coincide por país «Reino Unido» (+2). |
| 48 | R15 | Coincide el país | [C31] coincide por país «Reino Unido» (+2). |
| 49 | R15 | Coincide el país | [C25] coincide por país «Reino Unido» (+2). |
| 50 | R15 | Coincide el país | [C13] coincide por país «Reino Unido» (+2). |
| 51 | R15 | Coincide el país | [C09] coincide por país «Reino Unido» (+2). |
| 52 | R16 | Coincide el idioma | [C34] coincide por idioma «Inglés» (+2). |
| 53 | R16 | Coincide el idioma | [C33] coincide por idioma «Inglés» (+2). |
| 54 | R16 | Coincide el idioma | [C32] coincide por idioma «Inglés» (+2). |
| 55 | R16 | Coincide el idioma | [C31] coincide por idioma «Inglés» (+2). |
| 56 | R19 | Descarte por idioma | [C30] descartada: idioma «Español» distinto de «Inglés». |
| 57 | R16 | Coincide el idioma | [C26] coincide por idioma «Inglés» (+2). |
| 58 | R16 | Coincide el idioma | [C25] coincide por idioma «Inglés» (+2). |
| 59 | R19 | Descarte por idioma | [C18] descartada: idioma «Instrumental» distinto de «Inglés». |
| 60 | R16 | Coincide el idioma | [C14] coincide por idioma «Inglés» (+2). |
| 61 | R16 | Coincide el idioma | [C13] coincide por idioma «Inglés» (+2). |
| 62 | R16 | Coincide el idioma | [C11] coincide por idioma «Inglés» (+2). |
| 63 | R16 | Coincide el idioma | [C10] coincide por idioma «Inglés» (+2). |
| 64 | R16 | Coincide el idioma | [C09] coincide por idioma «Inglés» (+2). |
| 65 | R16 | Coincide el idioma | [C08] coincide por idioma «Inglés» (+2). |
| 66 | R16 | Coincide el idioma | [C07] coincide por idioma «Inglés» (+2). |
| 67 | R16 | Coincide el idioma | [C06] coincide por idioma «Inglés» (+2). |
| 68 | R35 | Coincide un subgénero vecino | [C34] coincide por subgénero «Heavy metal», vecino de «Hard rock» (+1). |
| 69 | R35 | Coincide un subgénero vecino | [C33] coincide por subgénero «Heavy metal», vecino de «Hard rock» (+1). |
| 70 | R35 | Coincide un subgénero vecino | [C08] coincide por subgénero «Grunge», vecino de «Hard rock» (+1). |
| 71 | R13 | Coincide un género relacionado | [C24] coincide por género «Rock», que incluye al estilo pedido «Hard rock» (+1). |
| 72 | R13 | Coincide un género relacionado | [C23] coincide por género «Rock», que incluye al estilo pedido «Hard rock» (+1). |
| 73 | R13 | Coincide un género relacionado | [C22] coincide por género «Rock», que incluye al estilo pedido «Hard rock» (+1). |
| 74 | R13 | Coincide un género relacionado | [C21] coincide por género «Rock», que incluye al estilo pedido «Hard rock» (+1). |
| 75 | R13 | Coincide un género relacionado | [C20] coincide por género «Rock», que incluye al estilo pedido «Hard rock» (+1). |
| 76 | R13 | Coincide un género relacionado | [C19] coincide por género «Rock», que incluye al estilo pedido «Hard rock» (+1). |
| 77 | R13 | Coincide un género relacionado | [C09] coincide por género «Rock», que incluye al estilo pedido «Hard rock» (+1). |
| 78 | R13 | Coincide un género relacionado | [C05] coincide por género «Rock», que incluye al estilo pedido «Hard rock» (+1). |
| 79 | R13 | Coincide un género relacionado | [C04] coincide por género «Rock», que incluye al estilo pedido «Hard rock» (+1). |
| 80 | R13 | Coincide un género relacionado | [C03] coincide por género «Rock», que incluye al estilo pedido «Hard rock» (+1). |
| 81 | R13 | Coincide un género relacionado | [C02] coincide por género «Rock», que incluye al estilo pedido «Hard rock» (+1). |
| 82 | R13 | Coincide un género relacionado | [C01] coincide por género «Rock», que incluye al estilo pedido «Hard rock» (+1). |

*Nivel 3: puntaje y decisión*

| Paso | Regla | Nombre | Descripción |
|---|---|---|---|
| 83 | R22 | Puntaje consolidado | [C09] puntaje consolidado = 7. |
| 84 | R22 | Puntaje consolidado | [C33] puntaje consolidado = 7. |
| 85 | R22 | Puntaje consolidado | [C31] puntaje consolidado = 9. |
| 86 | R22 | Puntaje consolidado | [C32] puntaje consolidado = 9. |
| 87 | R23 | Recomendación emitida | [C32] RECOMENDADA (puntaje 9 ≥ 3). |
| 88 | R23 | Recomendación emitida | [C31] RECOMENDADA (puntaje 9 ≥ 3). |
| 89 | R23 | Recomendación emitida | [C33] RECOMENDADA (puntaje 7 ≥ 3). |
| 90 | R23 | Recomendación emitida | [C09] RECOMENDADA (puntaje 7 ≥ 3). |

*Nivel 4: artistas y estilos sugeridos*

| Paso | Regla | Nombre | Descripción |
|---|---|---|---|
| 91 | R25 | Artista recomendado | Artista recomendado: Queen. |
| 92 | R26 | Estilo sugerido | Estilo sugerido: rock / rock progresivo. |
| 93 | R25 | Artista recomendado | Artista recomendado: Black Sabbath. |
| 94 | R26 | Estilo sugerido | Estilo sugerido: metal / heavy metal. |
| 95 | R25 | Artista recomendado | Artista recomendado: Led Zeppelin. |
| 96 | R26 | Estilo sugerido | Estilo sugerido: rock / hard rock. |
| 97 | R25 | Artista recomendado | Artista recomendado: Deep Purple. |

**Verificación del criterio de aceptación:**

- ✔ La traza tiene pasos numerados con regla y descripción
- ✔ Cubre lectura de la entrada, comparación, decisión y síntesis
- ✔ Cada recomendación trae motivos y cadena de reglas

### Resultado: ✅ OK

## Resumen de la ejecución

In [11]:
display(Markdown(tabla(
    [(c["id"], c["titulo"], c["prioridad"], c["requisitos"],
      "✅ OK" if resultados[c["id"]][1] else "❌ NO OK") for c in CASOS],
    ["Caso", "Objetivo", "Prioridad", "Requisitos", "Resultado"])))
aprobados = sum(ok for _, ok in resultados.values())
print(f"Casos aprobados: {aprobados} de {len(CASOS)}")

| Caso | Objetivo | Prioridad | Requisitos | Resultado |
|---|---|---|---|---|
| CA-01 | Recomendación por estilo y país (rock nacional) | MUST | RF-01 / RF-03 | ✅ OK |
| CA-02 | Artistas similares a una banda base | MUST | RF-02 / RF-08 | ✅ OK |
| CA-03 | Filtrado por época (años 80) | SHOULD | RF-04 | ✅ OK |
| CA-04 | Música local e idioma | SHOULD | RF-06 / RF-07 | ✅ OK |
| CA-05 | Consulta multiatributo | MUST | RF-01 / RF-03 / RF-05 | ✅ OK |
| CA-06 | Búsqueda sin coincidencias | Caso límite | RF-01 / RF-07 | ✅ OK |
| CA-07 | Entrada vacía | Caso límite | RF-01 | ✅ OK |
| CA-08 | Explicabilidad (sobre la consulta de CA-05) | MUST | RF-08 | ✅ OK |

Casos aprobados: 8 de 8


## Cobertura de la base de reglas

Reglas que se dispararon al menos una vez en esta batería de casos.

In [12]:
disparadas = {p["regla"] for r, _ in resultados.values() for p in r["traza"]}
display(Markdown(tabla(
    [(rid, nivel, nombre, "✔" if rid in disparadas else "—") for rid, (nivel, nombre) in sorted(REGLAS.items())],
    ["Regla", "Nivel", "Nombre", "Disparada"])))
print(f"{len(disparadas)} de {len(REGLAS)} reglas disparadas en los casos del cliente.")

| Regla | Nivel | Nombre | Disparada |
|---|---|---|---|
| R00 | 0 | Validación de datos mínimos | ✔ |
| R01 | 1A | Estilos pedidos | ✔ |
| R02 | 1A | Era convertida en período | ✔ |
| R03 | 1A | Año convertido en período (±2) | — |
| R04 | 1A | País pedido | ✔ |
| R05 | 1A | Música local como país local | ✔ |
| R06 | 1A | Idioma pedido | ✔ |
| R07 | 1B | Géneros afines | ✔ |
| R08 | 1B | Contexto latino por género | — |
| R09 | 1B | Contexto latino por país | ✔ |
| R10 | 1B | Contexto rock de los 80 | — |
| R11 | 2 | Coincide el género | ✔ |
| R12 | 2 | Coincide el subgénero | ✔ |
| R13 | 2 | Coincide un género relacionado | ✔ |
| R14 | 2 | Coincide la época | ✔ |
| R15 | 2 | Coincide el país | ✔ |
| R16 | 2 | Coincide el idioma | ✔ |
| R17 | 2 | Encaja en música latina | ✔ |
| R18 | 2 | Encaja en rock de los 80 | — |
| R19 | 2 | Descarte por idioma | ✔ |
| R20 | 2 | Descarte por época | ✔ |
| R21 | 2B | Descarte por estilo | ✔ |
| R22 | 3 | Puntaje consolidado | ✔ |
| R23 | 3 | Recomendación emitida | ✔ |
| R24 | 3 | Puntaje insuficiente | ✔ |
| R25 | 4 | Artista recomendado | ✔ |
| R26 | 4 | Estilo sugerido | ✔ |
| R27 | cierre | Sin coincidencia suficiente | ✔ |
| R28 | 1A | Artista de referencia | ✔ |
| R29 | 1A | Canción de referencia | — |
| R30 | 1B | Género que contiene al subgénero pedido | ✔ |
| R31 | 1B | Subgéneros vecinos del estilo pedido | ✔ |
| R32 | 1B | Rasgos del artista de referencia | ✔ |
| R33 | 1B | Rasgos de la canción de referencia | — |
| R34 | 1B | Estilos vecinos de la referencia | ✔ |
| R35 | 2 | Coincide un subgénero vecino | ✔ |
| R36 | 2 | Mismo estilo que la referencia | ✔ |
| R37 | 2 | Estilo vecino de la referencia | ✔ |
| R38 | 2 | Mismo idioma que la referencia | ✔ |
| R39 | 2 | Descarte: es el artista de referencia | ✔ |
| R40 | 2 | Descarte: es la canción de referencia | — |

34 de 41 reglas disparadas en los casos del cliente.
